In [0]:
%run ../log/00_audit_logging

In [0]:
from datetime import datetime
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, LongType, DoubleType, DateType, ArrayType
)
from delta.tables import DeltaTable

start_time = datetime.now()

# 1. Define Explicit Schema for Movie Fields inside raw_record
movie_schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("title", StringType(), True),
    StructField("original_title", StringType(), True),
    StructField("overview", StringType(), True),
    StructField("release_date", StringType(), True),
    StructField("popularity", DoubleType(), True),
    StructField("vote_average", DoubleType(), True),
    StructField("vote_count", LongType(), True),
    StructField("budget", LongType(), True),
    StructField("revenue", LongType(), True),
    StructField("runtime", IntegerType(), True),
    StructField("status", StringType(), True),
    StructField("tagline", StringType(), True)
])

try:
    # 2. Parse JSON & Cast Types
    df_bronze = spark.table("workspace.tmdb.bronze_movies").filter(f"batch_id = '{batch_id}'")

    df_silver_parsed = (
        df_bronze
        .withColumn("parsed", F.from_json(F.col("raw_record"), movie_schema))
        .select(
            F.col("parsed.id").alias("movie_id"),
            F.col("parsed.title").alias("title"),
            F.col("parsed.original_title").alias("original_title"),
            F.col("parsed.overview").alias("overview"),
            F.to_date(F.col("parsed.release_date"), "yyyy-MM-dd").alias("release_date"),
            F.col("parsed.popularity").cast(DoubleType()).alias("popularity"),
            F.col("parsed.vote_average").cast(DoubleType()).alias("vote_average"),
            F.col("parsed.vote_count").cast(LongType()).alias("vote_count"),
            F.col("parsed.budget").cast(LongType()).alias("budget"),
            F.col("parsed.revenue").cast(LongType()).alias("revenue"),
            F.col("parsed.runtime").cast(IntegerType()).alias("runtime"),
            F.col("parsed.status").alias("status"),
            F.col("parsed.tagline").alias("tagline"),
            F.col("source_collection"),
            F.col("batch_id"),
            F.current_timestamp().alias("load_timestamp")
        )
        .filter("movie_id IS NOT NULL") # Deduplicate / Filter corrupts
    )

    # 3. Create Silver Table if it doesn't exist
    spark.sql("""
    CREATE TABLE IF NOT EXISTS workspace.tmdb.silver_movies (
        movie_id INT,
        title STRING,
        original_title STRING,
        overview STRING,
        release_date DATE,
        popularity DOUBLE,
        vote_average DOUBLE,
        vote_count LONG,
        budget LONG,
        revenue LONG,
        runtime INT,
        status STRING,
        tagline STRING,
        source_collection STRING,
        batch_id STRING,
        load_timestamp TIMESTAMP
    )
    USING DELTA
    """)

    # 4. Perform Idempotent MERGE INTO (Upsert)
    target_table = DeltaTable.forName(spark, "workspace.tmdb.silver_movies")

    (
        target_table.alias("target")
        .merge(
            df_silver_parsed.alias("source"),
            "target.movie_id = source.movie_id"
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    record_count = df_silver_parsed.count()
    end_time = datetime.now()
    log_pipeline_execution("Silver", load_type, batch_id, start_time, end_time, "SUCCESS", record_count)

except Exception as e:
    end_time = datetime.now()
    log_pipeline_execution("Silver", load_type, batch_id, start_time, end_time, "FAILURE", 0, str(e))
    raise e